In [ ]:
import logging
import os
from typing import Dict, Optional
import numpy as np
import pandas as pd
import cupy as cp
import cupyx
import cupyx.scipy.sparse as cpx_sparse
import rapids_singlecell as rsc
import scanpy as sc  # plotting / AnnData utilities
import matplotlib as mpl
import time
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
mpl.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment

In [ ]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
def init_gpu_memory(managed_memory=False, pool_allocator=True, device_id=0):
    rmm.reinitialize(
        managed_memory=managed_memory,
        pool_allocator=pool_allocator,
        devices=device_id,
    )
    cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
def G_a(mu, sd):
    return mu**2 / sd**2


In [ ]:
def G_b(mu, sd):
    return mu / sd**2


In [ ]:
def subset_cells(adata, cells_per_category=5000, stratify_category_key='sample'):
    adata.obs['_cell_index'] = np.arange(adata.n_obs)
    subset_ind = []
    for ct in adata.obs[stratify_category_key].unique():
        ind = adata.obs[stratify_category_key] == ct
        subset_ind_ = adata.obs['_cell_index'][ind]
        n_samples = np.min((len(subset_ind_), cells_per_category))
        subset_ind += list(np.random.choice(subset_ind_, size=n_samples, replace=False))
    n_cells_subset = len(subset_ind)
    print(f'Subsetted adata from {adata.shape[0]} to {n_cells_subset} cells')
    return adata[subset_ind, :].copy()

In [ ]:
def rescale_distribution(dist: pd.DataFrame, n_factors: int):
    # `dist` is DataFrame or ndarray (cells x factors). We treat row-wise.
    dist = np.asarray(dist)
    q01 = np.quantile(dist, 0.01, axis=1).reshape((dist.shape[0], 1))
    mask = dist > q01
    dist = dist * mask + 0.01 - q01 * mask
    dist = (dist.T / dist.max(1)).T
    return dist

In [ ]:
def max_min_sampling(data: pd.DataFrame, n_waypoints: int):
    # identical to your version (CPU; tiny)
    waypoint_set = []
    no_iterations = int((n_waypoints) / data.shape[1])
    N = data.shape[0]
    for ind in data.columns:
        vec = np.ravel(data[ind])
        iter_set = [np.random.randint(0, N)]
        dists = np.zeros([N, no_iterations])
        dists[:, 0] = np.abs(vec - data[ind].values[iter_set])
        for k in range(1, no_iterations):
            min_dists = dists[:, 0:k].min(axis=1)
            new_wp = np.where(min_dists == min_dists.max())[0][0]
            iter_set.append(new_wp)
            dists[:, k] = np.abs(vec - data[ind].values[new_wp])
        waypoint_set += iter_set
    waypoints = data.index[waypoint_set].unique()
    return waypoints

In [ ]:
def _gpu_columnwise_mean_std(X_gpu_csr: cpx_sparse.csr_matrix):
    """
    Compute per-gene mean and std on GPU:
      mu = mean(X, axis=0)
      var = E[X^2] - (E[X])^2
    Returns cupy arrays (1D)
    """
    eps = 1e-8
    n_cells = X_gpu_csr.shape[0]
    # mean per col
    sums = cp.asarray(X_gpu_csr.sum(axis=0)).ravel()
    mu = sums / n_cells
    # E[X^2]
    X2 = X_gpu_csr.copy()
    X2.data = X2.data ** 2
    sums2 = cp.asarray(X2.sum(axis=0)).ravel()
    sq_mu = sums2 / n_cells
    var = sq_mu - mu ** 2
    std = cp.sqrt(cp.maximum(var, 0.0)) + eps
    return mu, std

In [ ]:
def align_plot_stability(fac1, fac2, name1, name2, align=True, return_aligned=False, title=''):
    corr12 = np.corrcoef(fac1, fac2, False)
    ind_top = np.arange(0, fac1.shape[1])
    ind_right = np.arange(0, fac2.shape[1]) + fac1.shape[1]
    corr12 = corr12[ind_top, :][:, ind_right]
    corr12[np.isnan(corr12)] = -1
    if align:
        assignment = linear_sum_assignment(2 - corr12)[1]
        img = corr12[:, assignment]
    else:
        assignment = np.arange(corr12.shape[1])
        img = corr12
    plt.imshow(img)
    plt.title(f"{title}\n{name1} vs {name2}")
    plt.xlabel(name2)
    plt.ylabel(name1)
    plt.tight_layout()
    if return_aligned:
        return corr12, assignment

# ---------- GPU versions of your major steps ----------

In [ ]:
def compute_pcs_knn_umap_gpu(
    adata_subset,
    fig_dir='',
    tech_category_key: Optional[str]=None,
    scale_max_value: int=10,
    n_comps: int=100,
    n_neighbors: int=15
    ):
    """
    GPU version of your 'compute_pcs_knn_umap':
        - log1p only (no normalize_total), matching your original
        - per-tech scaling (grouped z-score) implemented on GPU if tech_category_key is provided
        - RAPIDS PCA, drop PC1, RAPIDS neighbors+UMAP
    """
    # compute total_counts for plotting
    # Keep counts on GPU if possible
    rsc.get.anndata_to_GPU(adata_subset)
    # total_counts on GPU
    tc = cp.asarray(adata_subset.X.sum(axis=1)).ravel()
    adata_subset.obs['total_counts'] = cp.asnumpy(tc)
    # preserve raw counts in a layer
    adata_subset.layers['counts'] = adata_subset.X.copy()
    # log1p only (no normalize_total)
    rsc.pp.log1p(adata_subset)
    # scaling:
    if tech_category_key is None:
        # global scale on GPU
        rsc.pp.scale(adata_subset, max_value=scale_max_value)
    else:
        # grouped scale: compute group-wise mean/std on GPU and z-score
        cats = adata_subset.obs[tech_category_key].astype('category')
        adata_subset.obs[tech_category_key] = cats
        groups = cats.cat.categories.tolist()
        X_all = adata_subset.X  # csr on GPU
        # We'll write the scaled X back group-wise (in place)
        for g in groups:
            mask = (adata_subset.obs[tech_category_key].values == g)
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue
            Xg = X_all[idx, :].copy()
            mu, std = _gpu_columnwise_mean_std(Xg)
            # (X - mu) / std, clipped to scale_max_value
            # sparse affine transform: X - mu -> subtract per-column means from nonzeros
            # For efficiency, do dense-ish route in chunks if needed; here we do direct:
            Xg = Xg.tocoo(copy=False)
            # subtract per-col mu
            Xg.data = Xg.data - mu[cp.asarray(Xg.col)]
            # divide by std
            Xg.data = Xg.data / std[cp.asarray(Xg.col)]
            # clip
            Xg.data = cp.clip(Xg.data, a_min=-scale_max_value, a_max=scale_max_value)
            Xg = Xg.tocsr()
            X_all[idx, :] = Xg
        adata_subset.X = X_all
    # PCA on GPU
    rsc.tl.pca(adata_subset, n_comps=n_comps)
    # QC plot: PC1 vs total_counts (on CPU plot)
    plt.hist2d(
        adata_subset.obsm['X_pca'][:, 0].astype(float),
        adata_subset.obs['total_counts'].values.astype(float),
        bins=200, norm=mpl.colors.LogNorm()
    )
    plt.xlabel('PC 1'); plt.ylabel('Total RNA count')
    plt.savefig(os.path.join(fig_dir, 'NMF_init_PC1_total_counts.pdf')); plt.close()
    # drop PC1
    adata_subset.obsm['X_pca'] = adata_subset.obsm['X_pca'][:, 1:]
    adata_subset.varm['PCs'] = adata_subset.varm['PCs'][:, 1:]
    # neighbors + UMAP on GPU
    rsc.pp.neighbors(adata_subset, n_neighbors=n_neighbors)  # uses X_pca by default
    rsc.tl.umap(adata_subset, min_dist=0.2, spread=0.8)      # same params as your scanpy call
    return adata_subset

In [ ]:
def find_waypoint_gene_clusters_gpu(
    adata_neighbours,
    k='aver_norm',
    n_factors=300,
    margin_of_error=20,
    n_neighbors=15,
    labels_key=None,
    label_filter=None,
    verbose=True
):
    """
    GPU-based variant that uses PCs to represent genes (when labels_key is None).
    Builds a gene AnnData, runs GPU neighbors/UMAP for genes, then max–min waypoint selection.
    """
    if labels_key is not None:
        raise NotImplementedError("labels-based cluster averages not ported in this GPU version (can add if needed).")
    # Use PCs to represent genes, as in your original
    aver = pd.DataFrame(
        adata_neighbours.varm['PCs'],
        index=adata_neighbours.var_names,
        columns=[f'PC_{i+1}' for i in range(adata_neighbours.varm['PCs'].shape[1])],
    )
    gene_rates = {'aver': aver}
    gene_rates[k] = (gene_rates['aver'].T / gene_rates['aver'].abs().max(1)).T  # normalize by abs max per PC
    if verbose:
        print({kk: vv.shape for kk, vv in gene_rates.items()})
    # Build gene-level AnnData on CPU (lighter), but compute KNN/UMAP on GPU
    # We only need a tiny X so we don't move giant matrices unnecessarily.
    # Use tiny placeholder X and put representation into .obsm[k]
    gobs = adata_neighbours.var_names.copy()
    adata_neighbours_g = sc.AnnData(
        X=cp.asnumpy(cp.zeros((len(gobs), 1), dtype=cp.float32)),  # 1 dummy feature
        obs=pd.DataFrame(index=gobs)
    )
    # attach representation for neighbors
    adata_neighbours_g.obsm[k] = gene_rates[k].values
    # For coloring/size: log10 mean expression (from main adata) — compute on GPU, bring back
    rsc.get.anndata_to_GPU(adata_neighbours)
    X_cell_gene = adata_neighbours.X  # csr GPU
    mean_per_gene = cp.asnumpy(cp.asarray(X_cell_gene.mean(axis=0)).ravel())
    adata_neighbours_g.obs['total_counts'] = np.log10(mean_per_gene + 1e-8)
    # neighbors on GPU using the representation
    # rsc.pp.neighbors can use 'use_rep' that points to obsm key
    rsc.pp.neighbors(adata_neighbours_g, n_neighbors=n_neighbors, use_rep=k, metric='correlation')
    rsc.tl.umap(adata_neighbours_g, min_dist=0.1, spread=2.5)
    # max–min sampling for waypoints (CPU)
    X_pd = pd.DataFrame(
        adata_neighbours_g.obsm[k],
        columns=[f"{k}_{i}" for i in range(adata_neighbours_g.obsm[k].shape[1])],
        index=adata_neighbours_g.obs_names,
    )
    init_n_factors = n_factors
    waypoints = max_min_sampling(data=X_pd, n_waypoints=n_factors)
    total_steps = 0
    while (abs(len(waypoints) - n_factors) > margin_of_error) and (total_steps <= 10):
        if verbose:
            print(len(waypoints), init_n_factors)
        waypoints = max_min_sampling(data=X_pd, n_waypoints=int(round(init_n_factors)))
        init_n_factors += 1.0 * (n_factors - len(waypoints))
        total_steps += 1
    n_factors = len(waypoints)
    # annotate
    adata_neighbours_g.obs["is_waypoint"] = adata_neighbours_g.obs_names.isin(waypoints)
    adata_neighbours_g.obs["is_waypoint_size"] = np.array([10 if x else 1 for x in adata_neighbours_g.obs["is_waypoint"]])
    adata_neighbours_g.obs["is_waypoint"] = adata_neighbours_g.obs["is_waypoint"].astype("category")
    return adata_neighbours_g, n_factors

In [ ]:
def _ensure_gpu_csr(X):
    """Return X as cupyx.scipy.sparse.csr_matrix, handling scipy/cupy/dense."""
    import scipy.sparse as sp
    if isinstance(X, cpx_sparse.csr_matrix):
        return X
    if isinstance(X, cp.ndarray):
        return cpx_sparse.csr_matrix(X)
    if sp.isspmatrix(X):  # CPU scipy sparse
        X = X.tocsr()
        data   = cp.asarray(X.data)
        indices= cp.asarray(X.indices)
        indptr = cp.asarray(X.indptr)
        return cpx_sparse.csr_matrix((data, indices, indptr), shape=X.shape)
    if isinstance(X, np.ndarray):
        return cpx_sparse.csr_matrix(cp.asarray(X))
    # cupyx sparse view or other sparse types
    try:
        return cpx_sparse.csr_matrix(X)
    except Exception as e:
        raise TypeError(f"Unsupported X type for GPU CSR conversion: {type(X)}") from e


def compute_w_initial_waypoint_gpu(
    adata_neighbours,
    adata_neighbours_g,
    n_factors,
    fig_dir='',
    k='aver_norm',
    scale=False, tech_category_key=None,
    use_x=True, layer=None,
    knn_smoothing=True,
    scale_max_value=10
):
    """
    GPU version of your W initialisation:
      W_init = X @ K_gene^T (averaged over waypoint genes' KNN neighborhood)
      optional KNN smoothing over cells (using cell connectivities)
    """
    # Ensure both AnnData objects are on GPU
    rsc.get.anndata_to_GPU(adata_neighbours)
    rsc.get.anndata_to_GPU(adata_neighbours_g)
    # --- Choose X (cells x genes selected) and force to GPU CSR
    if use_x and layer is None:
        X = adata_neighbours[:, adata_neighbours_g.obs_names].X
    elif layer is not None:
        X = adata_neighbours[:, adata_neighbours_g.obs_names].layers[layer]
    else:
        X = adata_neighbours[:, adata_neighbours_g.obs_names].X
    X = _ensure_gpu_csr(X)
    # --- Waypoint indices (use integer indexing for sparse slicing)
    wp_mask = adata_neighbours_g.obs["is_waypoint"].values.astype(bool)
    wp_idx = np.where(wp_mask)[0]
    # --- Gene connectivities (genes x genes) → GPU CSR
    K_gene_raw = adata_neighbours_g.obsp["connectivities"]  # could be CPU or GPU
    K_gene = _ensure_gpu_csr(K_gene_raw)
    # Select waypoint rows (n_wp x n_genes) on GPU
    K_wp = K_gene[wp_idx, :]  # cupyx CSR slice is OK
    # Column-normalise by per-waypoint degree (row sums)
    denom = cp.asarray(K_wp.sum(axis=1)).ravel()
    denom = cp.where(denom == 0, 1.0, denom)
    # --- Core multiply on GPU:
    # X: (n_cells x n_genes), K_wp.T: (n_genes x n_wp) -> (n_cells x n_wp)
    K_wp_T = K_wp.T.tocsr()                 # ensure CSR on the RHS too
    W_init_gpu = (X @ K_wp_T).tocsr()
    # divide each column by its denom
    W_init_gpu = W_init_gpu.tocoo(copy=False)
    W_init_gpu.data = W_init_gpu.data / denom[cp.asarray(W_init_gpu.col)]
    W_init_gpu = W_init_gpu.tocsr()
    # --- Optional cell-KNN smoothing on GPU
    if knn_smoothing:
        K_cell_raw = adata_neighbours.obsp["connectivities"]  # (n_cells x n_cells)
        K_cell = _ensure_gpu_csr(K_cell_raw)
        denom_cell = cp.asarray(K_cell.sum(axis=1)).ravel()
        denom_cell = cp.where(denom_cell == 0, 1.0, denom_cell)
        W_init_gpu = (K_cell @ W_init_gpu).tocsr()
        W_init_gpu = W_init_gpu.tocoo(copy=False)
        W_init_gpu.data = W_init_gpu.data / denom_cell[cp.asarray(W_init_gpu.row)]
        W_init_gpu = W_init_gpu.tocsr()
    # --- Rescale and store
    W_dense = W_init_gpu.toarray()          # cupy dense
    W_dense_cpu = cp.asnumpy(W_dense)       # to CPU for rescaling/plotting
    W_rescaled = rescale_distribution(W_dense_cpu, n_factors=W_dense_cpu.shape[0]).astype(np.float32)
    w_init_df = pd.DataFrame(
        W_rescaled,
        index=adata_neighbours.obs_names,
        columns=[f'factor_{i}' for i in range(W_rescaled.shape[1])]
    )
    adata_neighbours.uns.setdefault('mod_init', {})
    adata_neighbours.uns['mod_init']['initial_values'] = {'w_init': {'cell_factors_w_cf': w_init_df}}
    plt.hist(W_rescaled.ravel(), bins=500)
    plt.xlabel('Cell loading'); plt.ylabel('Frequency')
    plt.savefig(os.path.join(fig_dir, 'histogram_init_cell_loadings.pdf')); plt.close()
    return adata_neighbours

In [ ]:
def find_stable_waypoint_gene_clusters_gpu(
    adata_neighbours,
    n_factors=300,
    fig_dir='',
    k='aver_norm',
    n_neighbors=20,
    labels_key='cell_type',
    n_repeats=5,
    cluster_max_cutoff=0.2,
    margin_of_error=20,
    bootstrap_p=0.9,
    verbose=True
):
    """
    GPU port of your stability bootstrapping (keeps CPU alignment/plots).
    """
    np.random.seed(1)
    adata_list = []
    for i in range(n_repeats):
        from sklearn.model_selection import train_test_split
        ind_, _ = train_test_split(
            np.arange(adata_neighbours.n_obs),
            test_size=1 - bootstrap_p,
            shuffle=True,
            stratify=adata_neighbours.obs[labels_key],
        )
        adata_sub = adata_neighbours[ind_, :].copy()
        adata_g, _ = find_waypoint_gene_clusters_gpu(
            adata_neighbours=adata_sub,
            k=k, n_factors=n_factors, margin_of_error=margin_of_error,
            n_neighbors=n_neighbors, labels_key=None, verbose=verbose
        )
        adata_list.append(adata_g.copy())
    # compute max correlation per cluster across bootstraps (CPU)
    cluster_max = np.zeros((adata_list[0].obs["is_waypoint"].values.astype(bool).sum()))
    for i in range(len(adata_list) - 1):
        c0 = np.array(adata_list[0].obsp["connectivities"][adata_list[0].obs["is_waypoint"].values.astype(bool), :].T.toarray())
        c1 = np.array(adata_list[i+1].obsp["connectivities"][adata_list[i+1].obs["is_waypoint"].values.astype(bool), :].T.toarray())
        corr01, assignment = align_plot_stability(
            fac1=c0, fac2=c1, name1='0', name2=f'{i+1}', title='Bootstrap step',
            align=True, return_aligned=True
        )
        plt.savefig(os.path.join(fig_dir, 'align_plot_stability.pdf')); plt.close()
        cluster_max = np.array([cluster_max, corr01.max(1)]).mean(0)
    plt.hist(cluster_max, bins=20)
    plt.savefig(os.path.join(fig_dir, 'cluster_max.pdf')); plt.close()
    waypoints = adata_list[0].obs_names[adata_list[0].obs["is_waypoint"].values.astype(bool)]
    waypoints = waypoints[cluster_max > cluster_max_cutoff]
    n_factors = len(waypoints)
    adata_neighbours_g = adata_list[0].copy()
    adata_neighbours_g.obs["is_waypoint"] = adata_neighbours_g.obs_names.isin(waypoints)
    adata_neighbours_g.obs["is_waypoint_size"] = np.array([10 if x else 1 for x in adata_neighbours_g.obs["is_waypoint"]])
    adata_neighbours_g.obs["is_waypoint"] = adata_neighbours_g.obs["is_waypoint"].astype("category")
    with mpl.rc_context({'figure.figsize': [6, 6]}):
        sns.scatterplot(
            x=adata_neighbours_g.obsm["X_umap"][:,0],
            y=adata_neighbours_g.obsm["X_umap"][:,1],
            hue=adata_neighbours_g.obs['is_waypoint'],
            s=adata_neighbours_g.obs['is_waypoint_size']
        )
        plt.savefig(os.path.join(fig_dir, 'scatter_X_umap_waypoint.pdf')); plt.close()
    return adata_neighbours_g, n_factors

In [ ]:
def find_initial_values(
    adata,
    n_factors: int,
    stratify_category_key: str,
    tech_category_key: Optional[str],
    fig_dir='',
    cells_per_category: int = 100000
) -> Dict[str, np.ndarray]:
    """
    GPU-accelerated drop-in replacement.
    Returns:
        {'cell_factors_w_cf': (n_cells, n_factors) float32} – same as your original.
    """
    logging.info('find_initial_values[gpu] : .obs_names_make_unique()')
    adata_neighbours = adata.copy()
    adata_neighbours.uns['mod'] = {'gene_names': np.array(adata.var.index)}
    adata_neighbours.obs_names_make_unique()
    # Move X (cell-gene count matrix) to GPU
    rsc.get.anndata_to_GPU(adata_neighbours)
    # --- Step 1.0: subset
    logging.info(f'find_initial_values[gpu] : subset_cells(cells_per_category={cells_per_category}, stratify="{stratify_category_key}")')
    np.random.seed(1)
    adata_subset = subset_cells(
        adata_neighbours,
        cells_per_category=cells_per_category,
        stratify_category_key=stratify_category_key
    )
    # Move subset to GPU
    rsc.get.anndata_to_GPU(adata_subset)
    adata_neighbours = adata_subset.copy()
    # --- Step 1.1: PCA/Neighbors/UMAP on GPU (drop PC1)
    logging.info('find_initial_values[gpu] : compute_pcs_knn_umap_gpu()')
    adata_subset = compute_pcs_knn_umap_gpu(
        adata_subset,
        tech_category_key=tech_category_key,
        scale_max_value=10,
        n_comps=n_factors,
        n_neighbors=25,
        fig_dir=fig_dir
    )
    # --- Step 2.0: waypoint gene clusters on GPU
    logging.info('find_initial_values[gpu] : find_waypoint_gene_clusters_gpu()')
    adata_subset_g, n_factors = find_waypoint_gene_clusters_gpu(
        adata_subset,
        k='aver_norm',
        n_factors=n_factors,
        margin_of_error=20,
        n_neighbors=10,
        labels_key=None,
        label_filter=None,
        verbose=True
    )
    # enforce CSR (GPU)
    # if not isinstance(adata_subset.X, cpx_sparse.csr_matrix):
    #     adata_subset.X = cpx_sparse.csr_matrix(adata_subset.X)
    # if not isinstance(adata_subset_g.X, cpx_sparse.csr_matrix):
    #     adata_subset_g.X = cpx_sparse.csr_matrix(adata_subset_g.X)
    # --- Step 3.0: compute W init via GPU KNN smoothing
    logging.info('find_initial_values[gpu] : compute_w_initial_waypoint_gpu()')
    # ensure same cell order
    adata_subset = adata_subset[adata_neighbours.obs_names, :].copy()
    adata_subset = compute_w_initial_waypoint_gpu(
        adata_subset,
        adata_subset_g,
        n_factors,
        scale=True,
        tech_category_key=tech_category_key,
        use_x=True,
        knn_smoothing=True,
        fig_dir=fig_dir
    )
    adata_neighbours.uns['mod_init'] = adata_subset.uns['mod_init'].copy()
    # materialize into .obs (like your original)
    adata_subset.obs = adata_subset.obs.copy()
    cf = adata_subset.uns['mod_init']['initial_values']['w_init']['cell_factors_w_cf']
    adata_subset.obs[cf.columns] = cf
    adata_subset.obs = adata_subset.obs.copy()
    # Final dict, aligned to full (subset) cell order, float32
    init_vals = adata_neighbours.uns['mod_init']['initial_values']['w_init']
    init_vals = {k: v.loc[adata_neighbours.obs_names, :].values.astype('float32') for k, v in init_vals.items()}
    return init_vals

In [ ]:
n_factors = 11
stratify_category_key = 'section'
tech_category_key = 'section'
cells_per_category=10000099

In [ ]:
# Load Anndata object
adata = sc.read_h5ad('/lustre/scratch124/cellgen/bayraktar/kr23/projects/STAGE/webatlas/output/NMF/STAGE__concatenated__raw_counts.h5ad')
rsc.get.anndata_to_GPU(adata)
genes = pd.read_csv('/nfs/team283/kr23/projects/STAGE/webatlas/data/NMF/STAGE_ASD_susceptbility_genes.csv',
                    header=None).iloc[:,0].tolist()
logging.info(f'Subsetting to #{len(genes)} genes : {genes}')
shp = adata.shape
adata = adata[:, [gene in genes for gene in adata.var_names]].copy()
logging.info(f'Reduced from shape {shp} to {adata.shape}')

# Initialise directory to write quality control figures
args_fig_dir = "/nfs/team283/kr23/github/Xenium_NMF/figs"
os.makedirs(args_fig_dir, exist_ok=True)

In [ ]:
t1 = time.time()
init_vals = find_initial_values(
    adata=adata,
    n_factors=n_factors,
    stratify_category_key=stratify_category_key,
    tech_category_key=tech_category_key,
    fig_dir=args_fig_dir,
    cells_per_category=cells_per_category,
)
elapsed = time.time() - t1
print(f"Initialisation done in {elapsed:.2f}s")

In [ ]:
import pickle
with open('/nfs/team283/kr23/github/Xenium_NMF/xenium_NMF/init_values.pkl', 'wb') as handle:
    pickle.dump(init_vals, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pickle
with open('/lustre/scratch124/cellgen/bayraktar/kr23/projects/STAGE/webatlas/output/NMF/STAGE__NMF_init_values_10_factors.pkl', 'rb') as handle:
    init_vals_big = pickle.load(handle)

Then switching to Xenium_NMF

In [ ]:
import argparse
import xenium_NMF
import numpy as np
import pandas as pd
import scanpy as sc
import time
import torch
import pyro
import pickle
import sys
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
# Load Anndata object
t_start = time.time()
adata = sc.read_h5ad('/lustre/scratch124/cellgen/bayraktar/kr23/projects/STAGE/webatlas/output/NMF/STAGE__concatenated__raw_counts.h5ad')
adata.obs_names_make_unique()
# Subset Anndata to genes if provided with one
genes = pd.read_csv('/nfs/team283/kr23/projects/STAGE/webatlas/data/NMF/STAGE_ASD_susceptbility_genes.csv',
                    header=None).iloc[:,0].tolist()
logging.info(f'Subsetting to #{len(genes)} genes : {genes}')
shp = adata.shape
adata = adata[:, [gene in genes for gene in adata.var_names]].copy()
logging.info(f'Reduced from shape {shp} to {adata.shape}')
# Read initialized cell loading values from pickle file
with open('/nfs/team283/kr23/github/Xenium_NMF/xenium_NMF/init_values.pkl', 'rb') as handle:
    init_vals = pickle.load(handle)

# Load number of NMF factors
n_factors = init_vals[list(init_vals.keys())[0]].shape[1]
logging.info(f'NMF : Number factors = {n_factors}')

In [ ]:

# Setup Anndata for NMF model
xenium_NMF.NMF_Model.setup_anndata(adata=adata,
                                    batch_key = 'section')
# Setup NMF model
mod_NMF = xenium_NMF.NMF_Model(adata,
                                n_factors=n_factors,
                                init_vals=init_vals)